In [24]:
import requests
import pandas as pd
from sqlalchemy import create_engine

In [25]:
url = "https://api.frankfurter.dev/v1/2026-01-01..2026-08-01?from=USD&to=EUR,GBP"

response = requests.get(url)
response.raise_for_status()

print(response.url)
print(response.status_code)
print(response.text[:500])

data = response.json()

https://api.frankfurter.dev/v1/2026-01-01..2026-08-01?from=USD&to=EUR,GBP
200
{"amount":1.0,"base":"USD","start_date":"2025-12-31","end_date":"2026-07-31","rates":{"2025-12-31":{"EUR":0.85106,"GBP":0.74264},"2026-01-02":{"EUR":0.85317,"GBP":0.74388},"2026-01-05":{"EUR":0.85734,"GBP":0.74383},"2026-01-06":{"EUR":0.85419,"GBP":0.73998},"2026-01-07":{"EUR":0.85587,"GBP":0.74153},"2026-01-08":{"EUR":0.85653,"GBP":0.74407},"2026-01-09":{"EUR":0.85896,"GBP":0.74532},"2026-01-12":{"EUR":0.85529,"GBP":0.74187},"2026-01-13":{"EUR":0.85807,"GBP":0.74309},"2026-01-14":{"EUR":0.8583,


In [26]:
df= pd.DataFrame(data["rates"]).T
df.index.name="Date"
df=df.reset_index()

print(df.head())

         Date      EUR      GBP
0  2025-12-31  0.85106  0.74264
1  2026-01-02  0.85317  0.74388
2  2026-01-05  0.85734  0.74383
3  2026-01-06  0.85419  0.73998
4  2026-01-07  0.85587  0.74153


In [27]:
df = df.melt(
    id_vars=["Date"],
    var_name="TargetCurrency",
    value_name="ExchangeRate"
)

df["BaseCurrency"] = "USD"
df["ExtractedAt"] = pd.Timestamp.now()

print(df.head())


         Date TargetCurrency  ExchangeRate BaseCurrency  \
0  2025-12-31            EUR       0.85106          USD   
1  2026-01-02            EUR       0.85317          USD   
2  2026-01-05            EUR       0.85734          USD   
3  2026-01-06            EUR       0.85419          USD   
4  2026-01-07            EUR       0.85587          USD   

                 ExtractedAt  
0 2026-10-10 17:01:38.870573  
1 2026-10-10 17:01:38.870573  
2 2026-10-10 17:01:38.870573  
3 2026-10-10 17:01:38.870573  
4 2026-10-10 17:01:38.870573  


In [28]:
df["Date"] = pd.to_datetime(df["Date"])
df["ExchangeRate"] = pd.to_numeric(df["ExchangeRate"], errors="coerce")

df = df.dropna(subset=["Date", "ExchangeRate"])
df = df.drop_duplicates(
    subset=["Date", "BaseCurrency", "TargetCurrency"]
)

print(df.info())
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 298 entries, 0 to 297
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            298 non-null    datetime64[us]
 1   TargetCurrency  298 non-null    str           
 2   ExchangeRate    298 non-null    float64       
 3   BaseCurrency    298 non-null    str           
 4   ExtractedAt     298 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(1), str(2)
memory usage: 11.8 KB
None
Date              0
TargetCurrency    0
ExchangeRate      0
BaseCurrency      0
ExtractedAt       0
dtype: int64


In [29]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["MonthName"] = df["Date"].dt.month_name()
df["YearMonth"] = df["Date"].dt.to_period("M").astype(str)

print(df.head())


        Date TargetCurrency  ExchangeRate BaseCurrency  \
0 2025-12-31            EUR       0.85106          USD   
1 2026-01-02            EUR       0.85317          USD   
2 2026-01-05            EUR       0.85734          USD   
3 2026-01-06            EUR       0.85419          USD   
4 2026-01-07            EUR       0.85587          USD   

                 ExtractedAt  Year  Month MonthName YearMonth  
0 2026-10-10 17:01:38.870573  2025     12  December   2025-12  
1 2026-10-10 17:01:38.870573  2026      1   January   2026-01  
2 2026-10-10 17:01:38.870573  2026      1   January   2026-01  
3 2026-10-10 17:01:38.870573  2026      1   January   2026-01  
4 2026-10-10 17:01:38.870573  2026      1   January   2026-01  


In [30]:
print("Total rows:", len(df))
print("Currencies:", df["TargetCurrency"].unique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Total rows: 298
Currencies: <StringArray>
['EUR', 'GBP']
Length: 2, dtype: str
Date range: 2025-12-31 00:00:00 to 2026-07-31 00:00:00


In [31]:
df.to_csv("exchange_rates.csv", index=False)

In [32]:
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-RJ9GPG8;"
    "DATABASE=CurrencyAnalytics;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as connection:
    print("Connected successfully!")

Connected successfully!


In [33]:
from sqlalchemy import text

dates_df = df[["Date", "Year", "Month", "MonthName", "YearMonth"]].drop_duplicates("Date").copy()
dates_df["DateKey"] = dates_df["Date"].dt.strftime("%Y%m%d").astype(int)
dates_df["Date"] = dates_df["Date"].dt.date

with engine.begin() as connection:
    for _, row in dates_df.iterrows():
        connection.execute(
            text("""
                IF NOT EXISTS (
                    SELECT 1 FROM DimDate WHERE DateKey = :DateKey
                )
                INSERT INTO DimDate
                    (DateKey, FullDate, Year, Month, MonthName, YearMonth)
                VALUES
                    (:DateKey, :Date, :Year, :Month, :MonthName, :YearMonth)
            """),
            row.to_dict()
        )

print("DimDate loaded successfully!")

DimDate loaded successfully!


In [34]:
from sqlalchemy import text

with engine.begin() as connection:
    for currency in df["TargetCurrency"].unique():
        connection.execute(
            text("""
                IF NOT EXISTS (
                    SELECT 1 FROM DimCurrency
                    WHERE CurrencyCode = :currency
                )
                INSERT INTO DimCurrency (CurrencyCode, CurrencyName)
                VALUES (
                    :currency,
                    CASE
                        WHEN :currency = 'EUR' THEN 'Euro'
                        WHEN :currency = 'GBP' THEN 'British Pound'
                    END
                )
            """),
            {"currency": currency}
        )

    for _, row in df.iterrows():
        connection.execute(
            text("""
                IF NOT EXISTS (
                    SELECT 1 FROM FactExchangeRate
                    WHERE DateKey = :DateKey
                      AND BaseCurrency = :BaseCurrency
                      AND TargetCurrency = :TargetCurrency
                )
                INSERT INTO FactExchangeRate (
                    DateKey, BaseCurrency, TargetCurrency,
                    ExchangeRate, ExtractedAt
                )
                VALUES (
                    :DateKey, :BaseCurrency, :TargetCurrency,
                    :ExchangeRate, :ExtractedAt
                )
            """),
            {
                "DateKey": int(row["Date"].strftime("%Y%m%d")),
                "BaseCurrency": row["BaseCurrency"],
                "TargetCurrency": row["TargetCurrency"],
                "ExchangeRate": float(row["ExchangeRate"]),
                "ExtractedAt": row["ExtractedAt"].to_pydatetime()
            }
        )

print("FactExchangeRate loaded successfully!")

FactExchangeRate loaded successfully!


In [35]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM FactExchangeRate")
    )
    print("Total records in SQL Server:", result.scalar())

Total records in SQL Server: 298


In [36]:
df.to_csv("exchange_rates.csv", index=False)